# Short-term memory

메모리는 이전 상호작용에 대한 정보를 기억하는 시스템입니다. (Multi-turn)

단기 메모리를 사용하면 애플리케이션이 단일 스레드 또는 대화 내에서 이루어진 이전 상호 작용을 기억할 수 있습니다.

> https://docs.langchain.com/oss/python/langchain/short-term-memory

| 구분 | 단기 메모리 (Short-term memory) | 장기 메모리 (Long-term memory) |
| :--- | :--- | :--- |
| **핵심 개념** | 단일 대화(세션) 내의 흐름과 맥락 유지 | 여러 세션에 걸친 사용자 정보 및 지식 보관 |
| **관리 범위** | **Thread-scoped** (특정 대화 스레드 ID에 묶임) | **Namespace-scoped** (사용자 ID 또는 앱 단위) |
| **저장 기간** | 현재 진행 중인 대화 세션 동안 유지 | 세션 종료 후에도 영구 저장 및 재사용 가능 |
| **주요 기술** | Checkpointers (상태 저장 및 체크포인트) | Store / Vector DB (전역 데이터 저장소) |
| **데이터 내용** | 최근 주고받은 메시지 목록 (Message History) | 사용자 프로필, 선호도, 과거 주요 사건, 지식 |
| **주요 목적** | "그거", "저번에" 같은 대화 맥락 파악 | 미래의 상호작용을 위한 개인화 및 지식 축적 |
| **비유** | **작업 기억**: 현재 대화 내용을 머릿속에 담기 | **장기 기억**: 과거의 경험이나 배운 지식을 저장 |

<br>

> https://docs.langchain.com/oss/python/concepts/memory#memory-overview

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
)

In [6]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 김일남이야."}]},
)

response

{'messages': [HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}, id='b2039189-400f-4a75-bad8-facb24d4e6e9'),
  AIMessage(content=[{'type': 'text', 'text': '반갑습니다, 김일남 님! 성함이 아주 인상적이고 정겨운 느낌이 드네요. \n\n오늘 하루는 어떻게 보내고 계신가요? 제가 무엇을 도와드릴까요?', 'extras': {'signature': 'EjQKMgEMOdbHC9SfhUi+O/8Jm07GDr7fYo1N6Se5BNpIDHys2Ttjhjgt3AgohD7VWHILiqER'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e0815-94f4-76a1-990c-cba323559d91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 45, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}})]}

In [7]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 뭐야?"}]},
)

response

{'messages': [HumanMessage(content='내 이름은 뭐야?', additional_kwargs={}, response_metadata={}, id='5443cc96-72a5-4c4c-9424-160e1b3d911d'),
  AIMessage(content=[{'type': 'text', 'text': '죄송하지만, 저는 사용자의 개인 정보를 기억하거나 알고 있지 않습니다. 따라서 사용자의 성함이 무엇인지 알 수 없습니다.\n\n혹시 이전에 말씀해주신 적이 있다면 기억하지 못하는 점 양해 부탁드립니다. 저에게 이름을 알려주시면 대화 중에 기억하도록 하겠습니다! 성함이 어떻게 되시나요?', 'extras': {'signature': 'EjQKMgEMOdbHYZe4Fq07HI+whlkJxJ/nezmxT8Mog8yUHLY6Dq1kQDcyijf1hPoTth+TComa'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e0815-d8ea-7aa1-9fb4-2d943c64b785-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 69, 'total_tokens': 75, 'input_token_details': {'cache_read': 0}})]}

In [8]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
    checkpointer=InMemorySaver(),
)

In [9]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 김일남이야."}]},
    {"configurable": {"thread_id": "1"}},  
)

response

{'messages': [HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}, id='f26345cc-5bf7-491d-acfe-d8d0df5266a6'),
  AIMessage(content=[{'type': 'text', 'text': '반갑습니다, 김일남 님! 성함이 아주 기억하기 좋고 힘찬 느낌이네요.\n\n오늘 제가 무엇을 도와드릴까요? 궁금한 점이 있거나, 도움이 필요하시면 언제든 말씀해 주세요!', 'extras': {'signature': 'EjQKMgEMOdbHiFFzVVeRWCtBOhIqdjc2iCab0hTOfjeLArVfMas8G4+BthOaKYbGrHBihdFD'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e0816-dd17-7a21-a3e1-e7eee4852d57-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 48, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}})]}

In [10]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 뭐야?"}]},
    {"configurable": {"thread_id": "1"}},  
)

response

{'messages': [HumanMessage(content='내 이름은 김일남이야.', additional_kwargs={}, response_metadata={}, id='f26345cc-5bf7-491d-acfe-d8d0df5266a6'),
  AIMessage(content=[{'type': 'text', 'text': '반갑습니다, 김일남 님! 성함이 아주 기억하기 좋고 힘찬 느낌이네요.\n\n오늘 제가 무엇을 도와드릴까요? 궁금한 점이 있거나, 도움이 필요하시면 언제든 말씀해 주세요!', 'extras': {'signature': 'EjQKMgEMOdbHiFFzVVeRWCtBOhIqdjc2iCab0hTOfjeLArVfMas8G4+BthOaKYbGrHBihdFD'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e0816-dd17-7a21-a3e1-e7eee4852d57-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 48, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='내 이름은 뭐야?', additional_kwargs={}, response_metadata={}, id='d229759a-8691-4f84-aee2-6966a0d6ba65'),
  AIMessage(content=[{'type': 'text', 'text': '방금 말씀해 주셨듯이, 당신의 이름은 **김일남** 님입니다! 기억하고 있어요. :)', 'extras': {'signatur

In [11]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 뭐야?"}]},
    {"configurable": {"thread_id": "2"}},  
)

response

{'messages': [HumanMessage(content='내 이름은 뭐야?', additional_kwargs={}, response_metadata={}, id='c7e85fcd-7b10-4882-8808-e3ae0502d621'),
  AIMessage(content=[{'type': 'text', 'text': '죄송하지만, 저는 사용자의 개인 정보를 기억하거나 알고 있지 않기 때문에 성함을 알지 못합니다. \n\n만약 제가 당신의 이름을 기억해주길 원하신다면, 이름을 알려주세요! 다음 대화부터는 그 이름으로 불러드리겠습니다.', 'extras': {'signature': 'EjQKMgEMOdbHAcAcxfoygOEoTXzXB7IczCyQ/olyjtaSDD5/KKdE9IbDfXs1SydMa6C9oJMJ'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e0817-5dd4-7ba2-9e62-0cf6e5ce4829-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 55, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}})]}

In [12]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': '죄송하지만, 저는 사용자의 개인 정보를 기억하거나 알고 있지 않기 때문에 성함을 알지 못합니다. \n\n만약 제가 당신의 이름을 기억해주길 원하신다면, 이름을 알려주세요! 다음 대화부터는 그 이름으로 불러드리겠습니다.', 'extras': {'signature': 'EjQKMgEMOdbHAcAcxfoygOEoTXzXB7IczCyQ/olyjtaSDD5/KKdE9IbDfXs1SydMa6C9oJMJ'}}]
